# Week 10 — Privacy-constrained learning

**Goal.** Quantify what privacy costs a measurement system, in accuracy, at several points on the tradeoff.

**Deliverable.** Privacy/utility tradeoff curves — the evidence base for any privacy-era measurement argument.

**Rough shape of the week.** 2h reading (CriteoPrivateAd + ARA docs) · 6h building · 1h write-up.

---
### Ground rules (they apply every week)

1. **Beat a dumb baseline or it didn't happen.** Logistic regression or the global mean.
   Log the baseline in the same table as the fancy model.
2. **Split by time, never at random.** `split.time_split` — and call
   `split.check_no_leakage` so the assertion, not your memory, enforces it.
3. **Log every run** with `registry.log_result(...)`, including the ones that lost.
   The losing runs are what make the write-up honest.
4. **Write the finding down** in this week's `README.md` while it is fresh.

### Reading

PDFs are in `papers/` next to this notebook — see `papers/README.md`.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from adslab import data, metrics, plots, split, registry, encoders, calibration

plots.use_style()
pd.set_option("display.width", 140, "display.max_columns", 60)
print("harness ready")

## The dataset was built for this

CriteoPrivateAd tags every feature with the privacy regime it survives in. That mapping
*is* the experiment design — train on progressively smaller unions of buckets and the
accuracy you lose is the accuracy privacy costs.

`features_not_available_*` are the third-party-cookie-era signals that are gone. Training
with them gives you the oracle; the gap to it is the price.

In [ ]:
pa = data.load_privatead(days=list(range(1, 11)))
print(f"{len(pa):,} rows, days {sorted(pa.day_int.unique())}")

groups = data.privatead_feature_groups(pa)
for b, cols in groups.items():
    print(f"{len(cols):3d} cols  {b}")
    print(f"          {data.PRIVACY_BUCKETS[b]}")

for lab in ("is_clicked", "is_click_landed", "is_visit"):
    print(f"{lab:18s} {pa[lab].mean():.4%}")

## 1. The feature-ablation ladder

Train the same model on each nested feature set:

1. contextual only — the floor, available to everyone forever
2. + non-constrained key-value
3. + bit-constrained (Protected-Audience-style budget)
4. + browser bit-constrained
5. + `features_not_available` — the third-party-cookie oracle

Split by `day_int`. Plot AUC against feature set. **That plot is the deliverable** and it
is the single most reusable artifact in this repo: it is the answer to "how much does
Privacy Sandbox actually cost?" backed by your own numbers.

In [ ]:
sp = split.time_split(pa, "day_int", train_frac=0.7, val_frac=0.1)
split.check_no_leakage(pa, sp, time_col="day_int")
print(sp)

ladder = {
    "ctx_only":      groups["features_ctx_not_constrained"],
    "+kv_free":      groups["features_ctx_not_constrained"] + groups["features_kv_not_constrained"],
    "+kv_bits":      groups["features_ctx_not_constrained"] + groups["features_kv_not_constrained"] + groups["features_kv_bits_constrained"],
    "+browser_bits": groups["features_ctx_not_constrained"] + groups["features_kv_not_constrained"] + groups["features_kv_bits_constrained"] + groups["features_browser_bits_constrained"],
    "oracle_3pc":    sum(groups.values(), []),
}
{k: len(v) for k, v in ladder.items()}

## 2. Noised aggregates

The Attribution Reporting API does not hand you rows — it hands you *noisy aggregates*
under a contribution budget. Simulate it: group conversions by some key (campaign x day),
add Laplace noise calibrated to $\varepsilon$, and see what survives.

The finding to chase: utility depends brutally on **how many keys you split the budget
across**. Same $\varepsilon$, ten times the keys, and each aggregate is drowned. Sweep
both $\varepsilon$ and key cardinality and plot the surface. Almost everyone who
discusses this only sweeps $\varepsilon$, and that is the less interesting axis.

Read `papers/attribution-reporting-api-AGGREGATE.md` for how the real budget works.

In [ ]:
def laplace_mechanism(counts, epsilon, sensitivity=1.0, rng=None):
    rng = rng or np.random.default_rng(0)
    return counts + rng.laplace(0, sensitivity / epsilon, size=np.shape(counts))

# TODO: sweep epsilon x n_keys, measure relative error of the noised aggregate

## 3. Learning from aggregates only

The hard version: you never see a per-impression label, only noisy group totals. Train a
model whose *predicted group sums* match the observed noisy sums.

This is a real technique (learning from label proportions) and it is what measurement
looks like when the row-level join is gone for good. Even a partial result here is worth
more than a polished version of section 2.

In [ ]:
# TODO: loss on aggregate predicted-vs-observed sums per key

## 4. The tradeoff curves

Assemble everything into the deliverable: accuracy against privacy strength, on the same
axes, for each mechanism. Mark where a real product decision would sit.

Then the paragraph that makes it useful: for each regime, *what measurement question can
still be answered, and which one can't?* Aggregate reporting can still tell you which
campaign won. It cannot tell you which user to bid on. Say so plainly.

In [ ]:
# fig, ax = plt.subplots()
# ... AUC vs feature regime; relative error vs epsilon
# print(plots.save(fig, 10, "privacy_utility_tradeoff"))

---
## Log the results

Every model you tried, including the baseline and including the failures. `notes` is the
one sentence you would say out loud about the run — future-you assembles the write-up
from these, so write it now while you still remember why the run mattered.

In [ ]:
# registry.log_result(
#     week=10,
#     model="lightgbm_hashed_2^18",
#     metrics=metrics.evaluate(y_test, p_test),
#     dataset="attribution",
#     params=dict(n_bits=18, num_leaves=63, lr=0.05),
#     notes="beats LR by 0.011 AUC; most of the gain is from cat3 x cat7 interactions",
# )

print(registry.to_markdown(week=10))

---
## Write it up

Open `README.md` in this folder and fill in the three sections. Keep it to a page.

- **What I built** — one paragraph, no code.
- **What the numbers say** — paste the table above; say which comparison is the honest one.
- **What surprised me** — the part worth reading. If nothing surprised you, you probably
  did not stress the model hard enough.

Then commit:

```bash
git add week10_* results/
git commit -m "week 10: <the finding, not the task>"
```